In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# from langchain_ollama import ChatOllama
# from langchain_anthropic import ChatAnthropic

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [16]:
MODEL = "gemini-3.1-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')



In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 핵심이 되는 2개의 LangChain 객체 준비하기: retriever와 llm

#### 잠깐, "temperature"에 대하여:
- 출력이 얼마나 다양해지는지를 조절합니다
- temperature가 0이면 출력이 예측 가능하다는 뜻입니다
- temperature가 높을수록 답변이 더 다양해집니다

어떤 사람들은 temperature를 '창의성'처럼 설명하지만, 정확한 표현은 아닙니다
- 실제로는 추론 과정에서 어떤 토큰이 선택되는지를 조절합니다
- temperature=0은 항상 확률이 가장 높은 토큰을 선택한다는 의미입니다
- temperature=1은 보통 확률이 10%인 토큰이 10%의 확률로 선택된다는 의미입니다

참고: temperature가 0이라고 해서 출력이 항상 동일하게 재현되는 것은 아닙니다. 랜덤 시드(random seed)도 함께 설정해야 합니다.
참고 2: 창의성을 원한다면 시스템 프롬프트를 활용하세요!

In [18]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model=MODEL,
    google_api_key=google_api_key
)

In [27]:
retriever.invoke("Avery는 누구인가요?")

[Document(id='320d07c2-d379-4618-8520-4c6c61f13cea', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.'),
 Document(id='8dc2a748-bf26-4ebd-8672-3ff821989808', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2018**: **Exceeds Expectations**  \n  Under Avery’s pivoted vision, Insurellm launched two new successful products that significantly increased market share.  \n\n- **2019**: **Meets Expectations**  \n  Steady growth, however, some team tensions led to a minor

In [20]:
llm.invoke("Avery는 누구인가요?")

AIMessage(content=[{'type': 'text', 'text': "'Avery'라는 이름은 매우 흔하기 때문에, 구체적으로 어떤 분야의 누구를 찾으시는지에 따라 답변이 달라질 수 있습니다. 가장 대표적인 몇 가지 경우를 정리해 드립니다.\n\n**1. 대중문화 속의 인물**\n*   **에이버리 불록 (Avery Bullock):** 애니메이션 시리즈 《아메리칸 대드(American Dad!)》에 등장하는 CIA 부국장 캐릭터입니다.\n*   **에이버리 (Avery):** 게임 《포켓몬스터》 시리즈(특히 '갑옷의 외딴섬' DLC)에 등장하는 사이킥 타입 포켓몬 트레이너입니다.\n\n**2. 실존 인물 (유명인)**\n*   **에이버리 브런디지 (Avery Brundage):** 제5대 국제올림픽위원회(IOC) 위원장을 지낸 미국의 스포츠 행정가입니다.\n*   **텍스 에이버리 (Tex Avery):** 《루니 툰》 시리즈의 캐릭터들(벅스 버니, 대피 덕 등)을 탄생시키거나 발전시킨 전설적인 애니메이터이자 감독입니다.\n*   **에이버리 윌슨 (Avery Wilson):** 미국의 가수이자 댄서로, 《더 보이스(The Voice)》 시즌 3에 출연하며 이름을 알렸습니다.\n\n**3. 기타**\n*   **에이버리 데니슨 (Avery Dennison):** 세계적으로 유명한 미국의 소재 과학 및 제조 기업입니다. 주로 라벨, 포장재, 접착제 등을 생산하며, 사무용품 브랜드로도 잘 알려져 있습니다.\n*   **이름으로서의 Avery:** 영어권에서 성별에 관계없이 쓰이는 중성적인 이름으로, 원래는 성씨(Surname)에서 유래했습니다.\n\n혹시 **특정 분야(예: 영화, 게임, 역사, 기업 등)**에서 보신 인물이나 대상인가요? 추가 정보를 주시면 더 정확하게 답변해 드릴 수 있습니다.", 'extras': {'signature': 'EnEKbwERTTIPRsxCf77e6aKVPchnAoIIINzOkhqqOLbnP7nSj/lo9ROgncV

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
당신은 Insurellm이라는 회사를 대표하는, 해박하고 친절한 어시스턴스입니다.
당신은 지금 사용자와 Insurellm에 대해 대화하고 있습니다.
관련이 있다면, 주어진 컨텍스트를 활용해 질문에 답하세요.
답을 모른다면 모른다고 말하세요.

컨텍스트:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [33]:
answer_question("Averi는 누구인가요?", [])


당신은 Insurellm이라는 회사를 대표하는, 해박하고 친절한 어시스턴스입니다.
당신은 지금 사용자와 Insurellm에 대해 대화하고 있습니다.
관련이 있다면, 주어진 컨텍스트를 활용해 질문에 답하세요.
답을 모른다면 모른다고 말하세요.

컨텍스트:
[Document(id='f0aba9da-d247-45f3-b6a5-cfc982471820', metadata={'source': 'knowledge-base\\employees\\Kevin Zhang.md', 'doc_type': 'employees'}, page_content='## Other HR Notes\n- **Education:** BS in Computer Science from UC Berkeley\n- **Skills:** Expert in Swift, Kotlin, React Native, mobile UI/UX patterns, App Store optimization\n- **Recognition:** Mobile Innovation Award 2023 for Android app launch\n- **Open Source:** Active contributor to mobile development open-source projects\n- **Speaking:** Presented at Mobile DevCon 2023 on cross-platform development strategies\n- **Feedback:** Highly skilled mobile developer with excellent product sense. Takes ownership of mobile platform and drives continuous improvement. Strong mentor to other developers.'), Document(id='f8de9c97-45cf-402b-a6fe-4983617e8389', metadata={'doc_type': 'employees', 'sour

''